In [6]:
import torch
import torch.nn as nn

1. You are training a classifier. After several steps, the loss becomes NaN.
```
logits = model(x)
loss = torch.log(torch.softmax(logits, dim=-1))
loss = loss.mean()
```

In [4]:
# Bug: Taking log(softmax) directly is numerically unstable.

```
loss = torch.nn.functional.cross_entropy(logits, targets)
```
or
```
loss = torch.nn.functional.log_softmax(logits, dim=-1).mean()
```

2. Your RNN training diverges with exploding gradients.
Add gradient clipping to this training loop.
```
loss.backward()
optimizer.step()

```


```
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
```

3. Fix the following loss function.

In [9]:
def loss_fn(p):
    return -torch.log(p).mean()

In [11]:
def loss_fn_stable(p, eps=1e-8):
    return -torch.log(p.clamp(min=eps)).eam()

4. This attention code sometimes produces NaNs. Why?
```
scores = Q @ K.T
weights = torch.softmax(scores, dim=-1)
```

In [ ]:
scores = scores / math.sqrt(Q.size(-1)) # scaling, to make the attention logits dimension-invariant
scores = scores - scores.max(dim=-1, keepdim=True).values # identity after softmax
weights = torch.softmax(scores, dim=-1)

Extra real-world NaN causes (common in interviews) Even with scaling + max-subtraction, NaNs can still happen if:

- Masking is wrong: e.g., you do scores.masked_fill(mask, -inf) and a row becomes all -inf → softmax becomes 0/0 → NaN.
Fix: ensure at least one valid key per query, or replace -inf with a large negative like -1e9 (dtype-aware), or handle fully-masked rows explicitly.

- Mixed precision overflow: fp16 has limited range; logits can blow up faster.
Fix: compute softmax in fp32: weights = softmax(scores.float(), dim=-1).type_as(scores)

- Upstream NaNs in Q/K from earlier layers (bad init, too high LR, no grad clipping, etc.).
Fix: check torch.isnan(Q).any() / K and apply training stabilizers.

5. This loss becomes NaN intermittently.
```
loss = F.cross_entropy(logits, targets)
```

In [13]:
# targets contains invalid indices, labels out of [0, num_class -1]

In [ ]:
assert targets.min() >= 0
assert targets.max() < logits.size(-1)

6. Training with FP16 causes NaNs. Fix it.

In [ ]:
# answer
scaler = torch.cuda.amp.GradScaler()

optimizer.zero_grad()
with torch.cuda.amp.autocast():
    outputs = model(x)
    loss = criterion(outputs, y)

scaler.scale(loss).backward()

# Optional but highly recommended
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

scaler.step(optimizer)
scaler.update()

7. Loss explodes immediately after a few steps.

In [ ]:
# answer
optimizer = AdamW(model.parameters(), lr=1e-5)

8. Loss becomes NaN when batch size is small.

In [18]:
d_model = 48
batch_norm = nn.BatchNorm1d(d_model)

In [24]:
# Only freeze parameters, running stats (running mean, variance) still updates
for param in batch_norm.parameters():
    param.requires_grad = False

In [ ]:
# go to freeze gradient notebook to know more about m.train() m.eval()

9. steps for debugging gradient explotion

- Check for NaNs in inputs
- Log gradient norms
- Reduce LR
- Enable gradient clipping
- Check loss formulation
- Check data normalization
- Freeze layers
- Inspect mixed precision behavior